# Mobile Money Agent Network: Data Cleaning and Quality Analysis

This notebook audits, cleans and analyses mobile-money agent transaction data. It identifies data-quality problems, applies reproducible cleaning rules, merges transactions with the agent register and produces county-level and weekly summaries.

In [2]:
!python generate_data.py

All datasets generated successfully!
Duka sales: 1000 rows
Agents: 20 rows
Agent transactions: 612 rows


In [3]:
import os

print(os.listdir("/content"))

['.config', 'generate_data.py', 'agents.csv', 'agent_transactions.csv', 'duka_sales.csv', 'sample_data']


In [4]:

import pandas as pd
import numpy as np

# Load the agent register
agents = pd.read_csv("agents.csv")

# Load transaction records
txns = pd.read_csv("agent_transactions.csv")

print("Agent register shape:", agents.shape)
print("Transaction data shape:", txns.shape)

print("\nAgent register - first 5 records:")
display(agents.head())

print("\nTransaction records - first 5 records:")
display(txns.head())


Agent register shape: (20, 3)
Transaction data shape: (612, 5)

Agent register - first 5 records:


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01



Transaction records - first 5 records:


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [5]:

print("AGENT DATA TYPES")
print(agents.dtypes)

print("\nAGENT MISSING VALUES")
print(agents.isna().sum())

print("\nTRANSACTION DATA TYPES")
print(txns.dtypes)

print("\nTRANSACTION MISSING VALUES")
print(txns.isna().sum())


AGENT DATA TYPES
agent_id     object
county       object
onboarded    object
dtype: object

AGENT MISSING VALUES
agent_id     0
county       0
onboarded    0
dtype: int64

TRANSACTION DATA TYPES
txn_id       object
agent_id     object
txn_type     object
amount      float64
txn_date     object
dtype: object

TRANSACTION MISSING VALUES
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64


In [6]:
# Check duplicate transactions
print("Duplicate transaction IDs:")
print(txns["txn_id"].duplicated().sum())

# Check transaction type variations
print("\nTransaction type values:")
print(txns["txn_type"].value_counts(dropna=False))

# Check transaction amount values
print("\nTransaction amount summary:")
print(txns["amount"].describe())

# Check unusual transaction amounts
print("\nLargest transaction amounts:")
print(txns["amount"].nlargest(10))

# Check date formats
print("\nSample transaction dates:")
print(txns["txn_date"].head(15))

# Check agent IDs that are not in the agent register
unknown_agents = txns.loc[
    ~txns["agent_id"].isin(agents["agent_id"]),
    "agent_id"
]

print("\nUnknown agent IDs:")
print(unknown_agents.value_counts())

Duplicate transaction IDs:
12

Transaction type values:
txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64

Transaction amount summary:
count       590.000000
mean       4537.201695
std       22836.165375
min         109.000000
25%        1086.750000
50%        1779.000000
75%        3069.750000
max      250000.000000
Name: amount, dtype: float64

Largest transaction amounts:
306    250000.0
315    250000.0
424    250000.0
463    250000.0
594    250000.0
183     28847.0
222     22316.0
573     16605.0
174     13362.0
212     12078.0
Name: amount, dtype: float64

Sample transaction dates:
0     2026-03-03
1     2026-04-27
2     2026-04-29
3     2026-04-24
4     2026-04-17
5     2026-04-23
6     2026-04-02
7     2026-04-12
8     2026-04-04
9     2026-04-17
10    2026-03-27
11    2026-04-05
12    2026-04-13
13    2026-04-07
14    2026-03-18
Name: txn_date, dtype: object

Unknown agent IDs:
agent_id
AG099    8
Name: 

In [7]:
# Count dates using different separators
slash_dates = txns["txn_date"].str.contains("/", na=False)

print("Dates using /:", slash_dates.sum())
print("\nExamples of dates using /:")
print(txns.loc[slash_dates, "txn_date"])

# Confirm missing amounts
print("\nMissing amount records:", txns["amount"].isna().sum())

Dates using /: 5

Examples of dates using /:
87     2026/04/27
128    2026/04/28
181    2026/03/20
437    2026/03/30
571    2026/03/04
Name: txn_date, dtype: object

Missing amount records: 22



## Task 10: Data Quality Audit

The transaction dataset was inspected for uniqueness, consistency, completeness, validity and referential integrity. The audit identifies each issue, the data-quality dimension affected, the relevant column and the number of affected rows.


In [8]:

# Count inconsistent transaction types
invalid_types = ~txns["txn_type"].isin(["Deposit", "Withdrawal"])

# Count duplicate transaction IDs
duplicate_count = txns["txn_id"].duplicated().sum()

# Count unknown agents
unknown_agent_count = (~txns["agent_id"].isin(agents["agent_id"])).sum()

# Count dates using slash separators
slash_date_count = txns["txn_date"].str.contains("/", na=False).sum()

# Calculate IQR outlier threshold
q1 = txns["amount"].quantile(0.25)
q3 = txns["amount"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

outlier_count = (txns["amount"] > upper_bound).sum()

# Build audit table
audit = pd.DataFrame([
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows_affected": duplicate_count
    },
    {
        "issue": "Inconsistent transaction type labels",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows_affected": invalid_types.sum()
    },
    {
        "issue": "Missing and invalid transaction amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows_affected": txns["amount"].isna().sum()
    },
    {
        "issue": "Non-standard date separators",
        "dimension": "Validity",
        "column": "txn_date",
        "rows_affected": slash_date_count
    },
    {
        "issue": "Transactions linked to unknown agents",
        "dimension": "Referential Integrity",
        "column": "agent_id",
        "rows_affected": unknown_agent_count
    },
    {
        "issue": "Amounts exceeding the IQR upper bound",
        "dimension": "Accuracy",
        "column": "amount",
        "rows_affected": outlier_count
    }
])

display(audit)
print("\nIQR upper bound:", upper_bound)


,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Inconsistent transaction type labels,Consistency,txn_type,384
2,Missing and invalid transaction amounts,Completeness,amount,22
3,Non-standard date separators,Validity,txn_date,5
4,Transactions linked to unknown agents,Referential Integrity,agent_id,8
5,Amounts exceeding the IQR upper bound,Accuracy,amount,40



IQR upper bound: 6044.25


In [9]:

%%writefile cleaning_pipeline.py

import pandas as pd
import numpy as np


def clean_agent_data(txns, agents):
    """
    Clean mobile-money agent transactions and merge agent information.

    Returns:
        clean_df: Cleaned transaction DataFrame.
        log: Dictionary recording the cleaning actions and affected rows.
    """

    # Work on copies to avoid changing the original datasets
    df = txns.copy()
    agent_data = agents.copy()

    log = {
        "original_transaction_rows": len(df)
    }

    # 1. Remove duplicate transaction IDs
    duplicate_count = df["txn_id"].duplicated().sum()
    df = df.drop_duplicates(subset="txn_id", keep="first").copy()
    log["duplicate_rows_removed"] = int(duplicate_count)

    # 2. Standardize transaction type labels
    original_types = df["txn_type"].copy()

    df["txn_type"] = (
        df["txn_type"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({
            "deposit": "Deposit",
            "withdrawal": "Withdrawal"
        })
    )

    log["transaction_types_standardized"] = int(
        (original_types != df["txn_type"]).sum()
    )

    # 3. Convert amounts to numeric values
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    missing_amounts = df["amount"].isna().sum()
    median_amount = df["amount"].median()

    df["amount_imputed"] = df["amount"].isna()

    df["amount"] = df["amount"].fillna(median_amount)

    log["missing_amounts_imputed"] = int(missing_amounts)
    log["imputation_value"] = float(median_amount)

    # 4. Identify outliers using the IQR method
    q1 = pd.to_numeric(txns["amount"], errors="coerce").quantile(0.25)
    q3 = pd.to_numeric(txns["amount"], errors="coerce").quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + (1.5 * iqr)

    df["is_outlier"] = df["amount"] > upper_bound

    log["iqr_upper_bound"] = float(upper_bound)
    log["outliers_flagged"] = int(df["is_outlier"].sum())

    # 5. Standardize transaction dates
    df["txn_date"] = df["txn_date"].astype("string").str.replace(
        "/", "-", regex=False
    )

    df["txn_date"] = pd.to_datetime(
        df["txn_date"], errors="coerce"
    )

    log["invalid_dates"] = int(df["txn_date"].isna().sum())

    # 6. Merge transactions with agent information
    df = df.merge(
        agent_data[["agent_id", "county"]],
        on="agent_id",
        how="left",
        validate="many_to_one"
    )

    log["unknown_agents"] = int(df["county"].isna().sum())

    # Format dates as ISO YYYY-MM-DD strings
    df["txn_date"] = df["txn_date"].dt.strftime("%Y-%m-%d")

    log["final_transaction_rows"] = len(df)

    return df, log


Writing cleaning_pipeline.py


## Task 11: Cleaning Transactions Using a Reusable Pipeline

The cleaning function is imported from cleaning_pipeline.py and applied to the original transaction and agent datasets. The function removes duplicate transactions, standardises transaction types, handles missing amounts, flags outliers, standardises dates and merges agent county information.

In [10]:
from cleaning_pipeline import clean_agent_data

# Run the cleaning pipeline
clean_df, cleaning_log = clean_agent_data(txns, agents)

print("Cleaning completed successfully!")

print("\nCleaning Log:")
for action, result in cleaning_log.items():
    print(f"{action}: {result}")

print("\nCleaned dataset shape:", clean_df.shape)

display(clean_df.head())

Cleaning completed successfully!

Cleaning Log:
original_transaction_rows: 612
duplicate_rows_removed: 12
transaction_types_standardized: 376
missing_amounts_imputed: 20
imputation_value: 1776.5
iqr_upper_bound: 6044.25
outliers_flagged: 40
invalid_dates: 0
unknown_agents: 7
final_transaction_rows: 600

Cleaned dataset shape: (600, 8)


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru


In [11]:
print("Cleaned dataset shape:")
print(clean_df.shape)

print("\nTransaction types:")
print(clean_df["txn_type"].value_counts())

print("\nMissing values:")
print(clean_df.isnull().sum())

print("\nOutlier records:")
print(clean_df["is_outlier"].value_counts())

print("\nImputed amount records:")
print(clean_df["amount_imputed"].value_counts())

print("\nUnknown agent records:")
print(clean_df[clean_df["county"].isna()][["txn_id", "agent_id", "county"]])

Cleaned dataset shape:
(600, 8)

Transaction types:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

Missing values:
txn_id            0
agent_id          0
txn_type          0
amount            0
txn_date          0
amount_imputed    0
is_outlier        0
county            7
dtype: int64

Outlier records:
is_outlier
False    560
True      40
Name: count, dtype: int64

Imputed amount records:
amount_imputed
False    580
True      20
Name: count, dtype: int64

Unknown agent records:
      txn_id agent_id county
55   TX00056    AG099    NaN
68   TX00069    AG099    NaN
77   TX00078    AG099    NaN
237  TX00238    AG099    NaN
267  TX00268    AG099    NaN
444  TX00445    AG099    NaN
557  TX00558    AG099    NaN



## Task 17: County-Level and Weekly Transaction Analysis

The cleaned transaction data is summarised by county and transaction type to identify transaction volumes, total values and average transaction values.

A second summary uses weekly transaction values to compare activity across counties. Transactions associated with unknown agents are retained under an "Unknown" county category so that no transaction value is silently excluded.


In [12]:

# Create a copy for summary analysis
summary_df = clean_df.copy()

# Retain unknown agents under an explicit category
summary_df["county"] = summary_df["county"].fillna("Unknown")

# County summary split by transaction type
county_summary = (
    summary_df
    .groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        number_of_transactions=("txn_id", "count"),
        average_value=("amount", "mean")
    )
    .reset_index()
)

county_summary["total_value"] = county_summary["total_value"].round(2)
county_summary["average_value"] = county_summary["average_value"].round(2)

print("COUNTY-LEVEL TRANSACTION SUMMARY")
display(county_summary)


COUNTY-LEVEL TRANSACTION SUMMARY


,county,txn_type,total_value,number_of_transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.58
1,Kiambu,Withdrawal,77050.5,27,2853.72
2,Kisumu,Deposit,436420.5,74,5897.57
3,Kisumu,Withdrawal,76374.5,37,2064.18
4,Mombasa,Deposit,148986.0,72,2069.25
5,Mombasa,Withdrawal,368592.0,61,6042.49
6,Nairobi,Deposit,543590.0,118,4606.69
7,Nairobi,Withdrawal,162036.0,62,2613.48
8,Nakuru,Deposit,184890.0,60,3081.50
9,Nakuru,Withdrawal,328594.5,39,8425.50


In [13]:

# Convert transaction dates to datetime
summary_df["txn_date"] = pd.to_datetime(summary_df["txn_date"])

# Extract the Monday date for each transaction's week
summary_df["week_start"] = (
    summary_df["txn_date"].dt.to_period("W").dt.start_time
)

# Weekly transaction value by county
weekly_pivot = pd.pivot_table(
    summary_df,
    index="county",
    columns="week_start",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

print("WEEKLY TRANSACTION VALUE BY COUNTY (KES)")
display(weekly_pivot)


WEEKLY TRANSACTION VALUE BY COUNTY (KES)


week_start,2026-02-23,2026-03-02,2026-03-09,2026-03-16,2026-03-23,2026-03-30,2026-04-06,2026-04-13,2026-04-20,2026-04-27
county,,,,,,,,,,
Kiambu,0.0,18317.0,267771.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28125.0,27077.0,41077.0,15341.0,29416.0,37439.0,21308.0
Mombasa,6759.0,285669.5,30477.5,25144.5,17050.0,38344.0,47092.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52860.5,43107.5,41373.5,282931.0,56786.5,56432.5,65187.0,30934.5
Nakuru,2743.5,56574.0,28730.0,39795.0,18998.5,18145.0,276609.0,26152.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0


In [14]:
print("Total cleaned transaction value:",
      clean_df["amount"].sum())

print("Total county summary value:",
      county_summary["total_value"].sum())

print("Total weekly pivot value:",
      weekly_pivot.to_numpy().sum())

Total cleaned transaction value: 2688427.0
Total county summary value: 2688427.0
Total weekly pivot value: 2688427.0


## Task 18: Post-Merge Validation

The cleaned transactions were merged with the agent records using `agent_id`. A left join was used to preserve every valid transaction, including transactions linked to agent IDs that were not found in the agent master table.

The following checks confirm that the merge preserved the cleaned transaction count and identify records with unmatched agent information.

In [15]:
print("POST-MERGE VALIDATION")
print("---------------------")
print("Original transaction rows:", len(txns))
print("Rows after duplicate removal:", len(txns.drop_duplicates(subset="txn_id")))
print("Rows after agent merge:", len(clean_df))
print("Transactions with unknown agents:", clean_df["county"].isna().sum())
print("Unique transaction IDs:", clean_df["txn_id"].nunique())

POST-MERGE VALIDATION
---------------------
Original transaction rows: 612
Rows after duplicate removal: 600
Rows after agent merge: 600
Transactions with unknown agents: 7
Unique transaction IDs: 600



# Data Quality Report: Mobile Money Agent Network

## 1. Executive Overview

The mobile-money transaction dataset initially contained 612 records and was linked to an agent register containing 20 agents. The audit identified issues involving uniqueness, completeness, consistency, validity, accuracy and referential integrity. After cleaning, 600 unique transactions remained. The final cleaned transaction value was KES 2,688,427.

## 2. Data Quality Findings and Actions

### Uniqueness
The transaction extract contained 12 duplicate transaction IDs. These duplicates were removed using `drop_duplicates()`, retaining the first occurrence of each transaction ID. The final dataset contains 600 unique transactions.

### Completeness
There were 22 missing or invalid transaction amounts in the original dataset. After removing duplicate records, 20 missing amounts remained. These were imputed using the median transaction amount of KES 1,776.50. An `amount_imputed` flag was added to identify every imputed record.

### Consistency
The transaction type column contained five different representations of deposits and withdrawals, including inconsistent capitalization and whitespace. These were standardized into exactly two categories: `Deposit` and `Withdrawal`.

### Validity
Five transaction dates used slash separators instead of hyphens. The date values were standardized and converted into valid datetime values before being exported in ISO format (`YYYY-MM-DD`). No invalid dates remained after cleaning.

### Accuracy
The IQR method produced an upper outlier threshold of KES 6,044.25, identifying 40 suspicious transactions. The five KES 250,000 transactions were investigated as potential anomalies. They were retained and flagged rather than automatically deleted because high values may represent genuine transactions requiring business verification. An `is_outlier` flag was added.

### Referential Integrity
Eight original transactions referenced agent ID `AG099`, which was absent from the agent register. After duplicate removal, seven such transactions remained. A left join preserved all 600 transactions, while unmatched county values were left missing and identified for investigation.

## 3. Summary and Validation

The final dataset contains 600 unique transactions with standardized transaction types, numeric amounts, ISO-formatted dates and agent county information where available.

The county summary, weekly pivot and cleaned transaction dataset all produced a total transaction value of KES 2,688,427, confirming that the aggregations preserved the transaction values.

## 4. Privacy and Data Protection

If the extract also contained customer phone numbers, these would constitute personal data under the Kenya Data Protection Act, 2019. Such information would require lawful and transparent processing, collection limitation, purpose limitation and appropriate security safeguards. Phone numbers should be excluded from analytical outputs unless genuinely required, masked or pseudonymised where possible, encrypted when stored or transmitted, and accessible only to authorised personnel. The organisation should establish appropriate retention periods, protect the data against unauthorised access or disclosure, and respect the rights of data subjects.

## 5. Recommendations

- Investigate the seven transactions associated with unknown agent AG099 and reconcile them with the official agent register.
- Verify the five KES 250,000 transactions against source records before treating them as confirmed business activity.
- Maintain automated validation rules for transaction IDs, transaction types, amounts and dates.
- Monitor missing values and duplicate records during future data imports.
- Retain the cleaning log and quality flags to support transparency and future audits.


In [16]:
import os

os.makedirs("outputs", exist_ok=True)

clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)

print("Export successful:", os.path.exists("outputs/agent_transactions_clean.csv"))
print("Dataset shape:", pd.read_csv("outputs/agent_transactions_clean.csv").shape)

display(pd.read_csv("outputs/agent_transactions_clean.csv").head())

Export successful: True
Dataset shape: (600, 8)


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru


In [17]:
from google.colab import files

files.download("outputs/agent_transactions_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
files.download("cleaning_pipeline.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>